# Phase 2: stage the nationality backfill

Freezes the population (invest customer, `NationalityCountryCode__pc` empty in SF,
nationality available in the `stg_imp_investors_*` staging tables via
`external_id = ExternalID__pc`) into `crm_imp_person_accounts`, one row per account,
`_operation='update'`. Source priority `formated` -> `formated_2026070` ->
`formated_20260722` -> `w9_20260722` via COALESCE — zero conflicts and zero duplicate
matches verified, so the order is cosmetic (see `02_recon_nationality.sql`).

**This notebook only writes to the local MySQL staging table. It never touches Salesforce.**

Background: the investor imports never sent `nationality_country_code` — the
`imp_investors_*.sql` INSERT column lists omitted it, so every batch pushed without it.
Backfill only where the SF field is empty; codes are ISO-2 and all already live in the
SF picklist.

Prerequisites: mirrors refreshed via `camping-grubhof-import/refresh_sf_mirrors.py`,
then `01_create_mirror_indexes.sql` re-applied (the refresh drops the indexes).
Recon numbers (2026-08-25 mirror): 2,761 missing, expected population 2,669,
92 unmatched (review export, section 5).

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # repo root: config, mysql_client

from config import load_mysql_config
from mysql_client import MySQLClient

BATCH_ID = "2026-08-25_nationality_backfill"

db = MySQLClient(load_mysql_config())
print("connected | batch:", BATCH_ID)

## 1. Mirror freshness

`MAX(LastModifiedDate)` should be within a day of the refresh; if it is stale, stop and
re-run the refresh before staging. (The loader has a live skip-check as the last line
of defense, but staging fresh keeps the numbers honest.)

In [ ]:
row = db.fetch_one(
    "SELECT COUNT(*) AS n, MAX(LastModifiedDate) AS newest FROM crm_person_account_sfid_prod"
)
print(f"crm_person_account_sfid_prod  {row['n']:>12,}  newest: {row['newest']}")

## 2. Population counts

Expected from the 2026-08-25 recon: 6,158 invest customers, 2,761 missing nationality,
2,669 covered by the staging join. Small drift after a refresh is possible —
investigate anything larger than a handful (this population barely moves).

In [ ]:
# The exact phase-2 selection from 02_recon_nationality.sql query 5.
POPULATION_JOIN = """
    FROM crm_person_account_sfid_prod acc
    LEFT JOIN stg_imp_investors_formated s1
           ON s1.external_id = acc.ExternalID__pc AND s1.nationality_country_code <> ''
    LEFT JOIN stg_imp_investors_formated_2026070 s2
           ON s2.external_id = acc.ExternalID__pc AND s2.nationality_country_code <> ''
    LEFT JOIN stg_imp_investors_formated_20260722 s3
           ON s3.external_id = acc.ExternalID__pc AND s3.nationality_country_code <> ''
    LEFT JOIN stg_imp_investors_w9_20260722 s4
           ON s4.external_id = acc.ExternalID__pc
          AND s4.nationality_country_code IS NOT NULL AND s4.nationality_country_code <> ''
    WHERE acc.InvestCustomer__pc = 'True'
      AND (acc.NationalityCountryCode__pc IS NULL OR acc.NationalityCountryCode__pc = '')
"""
NATIONALITY_EXPR = """COALESCE(s1.nationality_country_code, s2.nationality_country_code,
                              s3.nationality_country_code, s4.nationality_country_code)"""

counts = db.fetch_one(f"""
    SELECT COUNT(*) AS missing,
           SUM({NATIONALITY_EXPR} IS NOT NULL) AS covered
    {POPULATION_JOIN}
""")
expected_accounts = int(counts["covered"])
print(f"missing nationality: {int(counts['missing']):,}")
print(f"covered by staging:  {expected_accounts:,}")
print(f"unmatched (review):  {int(counts['missing']) - expected_accounts:,}")

## 3. Stage the batch

One row per account. Guarded: refuses to run if the batch id already has rows — rerun
after a mistake means deleting the batch first, deliberately, not re-executing the cell.

Also adds the `_nationality_processed_at` bookkeeping column if the table does not have
it yet (the generic `_processed_at` is claimed by the older update scripts,
`_billing_processed_at` by the billing backfill).

In [ ]:
col = db.fetch_one("""
    SELECT COUNT(*) AS n FROM information_schema.columns
    WHERE table_schema = DATABASE()
      AND table_name = 'crm_imp_person_accounts'
      AND column_name = '_nationality_processed_at'
""")["n"]
if not col:
    db.execute("ALTER TABLE crm_imp_person_accounts ADD COLUMN _nationality_processed_at DATETIME NULL")
    print("column _nationality_processed_at added")
else:
    print("column _nationality_processed_at exists")

existing = db.fetch_one(
    "SELECT COUNT(*) AS n FROM crm_imp_person_accounts WHERE _batch_id = %s",
    (BATCH_ID,),
)["n"]
assert existing == 0, f"{existing} rows already staged under this batch id — not re-inserting"

inserted = db.execute(f"""
    INSERT INTO crm_imp_person_accounts
        (_operation, _batch_id, _excluded, source, last_name, email,
         sf_account_id, sf_person_contact_id,
         external_id, nationality_country_code)
    SELECT
        'update', %s, 0, 'nationality_backfill',
        acc.LastName, acc.PersonEmail,
        acc.Id, acc.PersonContactId,
        acc.ExternalID__pc, {NATIONALITY_EXPR}
    {POPULATION_JOIN}
      AND {NATIONALITY_EXPR} IS NOT NULL
""", (BATCH_ID,))

print(f"staged {inserted:,} rows under {BATCH_ID}")
assert inserted == expected_accounts, f"staged {inserted} but section 2 predicted {expected_accounts}"

## 4. Verification and predicted after-state

These numbers are the load contract: the dry-run must produce exactly this many rows,
and after the load the live count of invest customers without nationality should drop
from 2,761 to ~92 (the unmatched) plus whatever the live skip-check left out.

In [ ]:
summary = db.fetch_one("""
    SELECT COUNT(*) AS rows_staged,
           COUNT(DISTINCT sf_account_id) AS accounts,
           SUM(nationality_country_code IS NULL OR nationality_country_code = '') AS empty_code,
           COUNT(DISTINCT nationality_country_code) AS distinct_codes
    FROM crm_imp_person_accounts
    WHERE _batch_id = %s
""", (BATCH_ID,))
for k, v in summary.items():
    print(f"{k:15s} {int(v):,}")

codes = db.fetch_all("""
    SELECT nationality_country_code, COUNT(*) AS n
    FROM crm_imp_person_accounts
    WHERE _batch_id = %s
    GROUP BY 1 ORDER BY n DESC
""", (BATCH_ID,))
print("\ncode distribution:")
for c in codes:
    print(f"  {c['nationality_country_code']}: {int(c['n']):,}")

assert int(summary["rows_staged"]) == int(summary["accounts"]), "duplicate sf_account_id staged"
assert int(summary["empty_code"]) == 0, "rows without a nationality code staged"
assert all(len(str(c["nationality_country_code"])) == 2 for c in codes), "non-ISO-2 code staged"
print("\none row per account, every row has an ISO-2 code — staging frozen")

## 5. Review export (no writes): invest customers with no staging match

The ~92 invest customers whose `ExternalID__pc` matches none of the four staging
tables are NOT part of the backfill — there is simply no source value for them.
Exported for a human review; PII, lands in the gitignored `local_data/`.

In [ ]:
review = db.fetch_df(f"""
    SELECT acc.Id, acc.PersonEmail, acc.ExternalID__pc, acc.SourceSystem__pc,
           acc.CreatedDate, acc.InvestmentStatus__pc
    {POPULATION_JOIN}
      AND {NATIONALITY_EXPR} IS NULL
""")
out = Path.cwd().parent / "local_data" / "nationality_unmatched_review.csv"
out.parent.mkdir(parents=True, exist_ok=True)
review.to_csv(out, index=False)
print(f"{len(review):,} unmatched invest customers exported to {out}")

## Next

- Dry-run: `python nationality-backfill/update_nationality.py <batch_id> --dry-run`
  (or via `04_run_nationality_backfill.ipynb`, which drives all of phases 3-6).
- Probe: one account in prod, UI eyeball (04, `RUN_PROBE` gate).
- Bulk load: **does not run without Arsal's explicit go-ahead** (04, `RUN_LOAD` gate).